# Phase 6: Case Study Selection & Analysis (Orchestrator)

Selects representative samples, **automatically generates any missing XAI artifacts**
by calling Phases 2-5 on-demand, then creates combined multi-method explanation panels.

| Component | Description |
|---|---|
| Mode | Orchestrator — loads model, generates missing artifacts |
| Input | Prediction CSVs + dataset + existing artifacts |
| Auto-generation | Grad-CAM, Attention, SHAP, LIME (if missing) |
| Output | Combined figures (12×14, 300 DPI) + metadata + analysis |

**Case types:** correct, high_error, conflict, text_dominant, image_dominant, difficult, agreement

---
### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone and install

In [ ]:
!rm -rf /content/SE365
!git clone -b xai-v3 https://github.com/lechihoang/SE365.git /content/SE365
%cd /content/SE365
!pip install -q -r requirements.txt
!pip install -q shap lime scikit-image

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configuration

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

DRIVE_ROOT   = '/content/drive/MyDrive/SE365'
PROJECT_ROOT = '/content/SE365'
EXP_ID       = 'EXP_060A_bestsequential_full_configuration'

EXP_DIR      = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
XAI_DIR      = f'{EXP_DIR}/xai'
XAI_OUT_DIR  = f'{EXP_DIR}/xai/case_studies'
DATA_DIR     = f'{PROJECT_ROOT}/data/text'
IMAGE_DIR    = f'{PROJECT_ROOT}/data/image'

os.makedirs(XAI_OUT_DIR, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'EXP_DIR      : {EXP_DIR}')
print(f'XAI_OUT_DIR  : {XAI_OUT_DIR}')

### STEP 5: Imports and Seed

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 5 — Imports and Seed')
print('='*60)

import json, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage

from xai.config import (
    TARGET_NAMES, FACTOR_NAMES, DISPLAY_NAMES, NUM_TARGETS,
    DEFAULT_SEED, DEFAULT_DPI, THESIS_DPI,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL,
)
from xai.utils import (
    get_device, set_seed, get_tokenizer, get_image_processor,
    load_model, load_single_sample, save_raw_values,
)
from xai.case_study import (
    CaseStudyRunner, XAIOrchestrator, check_sample_artifacts,
)

SEED = DEFAULT_SEED
set_seed(SEED)
device = get_device()

print(f'Device : {device}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 6: Load Model + Tokenizer + Image Processor

Phase 6 now loads the model so it can generate missing XAI artifacts on-demand.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 6 — Load Model')
print('='*60)

model, config = load_model(EXP_DIR, device=device)

text_model_name = config.get('text_model_name', BEST_TEXT_MODEL)
image_model_name = config.get('image_model_name', BEST_IMAGE_MODEL)
tokenizer = get_tokenizer(text_model_name)
image_processor = get_image_processor(image_model_name)

print(f'Model : {model.__class__.__name__} ({sum(p.numel() for p in model.parameters()):,} params)')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 7: Determine Data Split and Check Artifacts

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 7 — Data Split + Artifact Check')
print('='*60)

test_pred = os.path.join(EXP_DIR, 'test_predictions.csv')
val_pred = os.path.join(EXP_DIR, 'predictions.csv')
SPLIT = 'test' if os.path.isfile(test_pred) else 'validation'
DATASET_CSV = os.path.join(DATA_DIR, 'test.csv' if SPLIT == 'test' else 'val.csv')
if not os.path.isfile(DATASET_CSV):
    DATASET_CSV = os.path.join(DATA_DIR, 'val.csv')

pred_df = pd.read_csv(test_pred if SPLIT == 'test' else val_pred)
dataset_df = pd.read_csv(DATASET_CSV)
error_cols = [c for c in pred_df.columns if c.startswith('absolute_error_')]
if error_cols:
    pred_df['mean_error'] = pred_df[error_cols].mean(axis=1)

print(f'Split       : {SPLIT}')
print(f'Predictions : {len(pred_df)} samples')
print(f'Dataset     : {len(dataset_df)} samples')

# Artifact diagnostic
n = min(20, len(pred_df), len(dataset_df))
art_counts = {'gradcam': 0, 'attention': 0, 'shap': 0, 'lime': 0}
print(f'\n{"sample_id":<16s} {"GC":>4s} {"AT":>4s} {"SH":>4s} {"LM":>4s}')
for i in range(n):
    sid = f'sample_{i:04d}'
    a = check_sample_artifacts(sid, XAI_DIR)
    for p in art_counts: art_counts[p] += int(a[p])
    gc = 'Y' if a['gradcam'] else '-'
    at = 'Y' if a['attention'] else '-'
    sh = 'Y' if a['shap'] else '-'
    lm = 'Y' if a['lime'] else '-'
    print(f'{sid:<16s} {gc:>4s} {at:>4s} {sh:>4s} {lm:>4s}')

print(f'\nExisting: GC={art_counts["gradcam"]}, AT={art_counts["attention"]}, '
      f'SH={art_counts["shap"]}, LM={art_counts["lime"]} (of {n} checked)')
print(f'Missing artifacts will be generated automatically by the orchestrator.')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 8: Prepare SHAP Background (for on-demand generation)

If SHAP artifacts need to be generated, we need a background embedding set.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 8 — SHAP Background')
print('='*60)

# Try to load existing background from Phase 4
bg_path = os.path.join(XAI_DIR, 'shap', 'raw', 'background_fused.pt')
shap_background = None

if os.path.isfile(bg_path):
    shap_background = torch.load(bg_path, map_location='cpu')
    print(f'Loaded existing SHAP background: {shap_background.shape}')
else:
    print('No existing SHAP background found.')
    print('SHAP artifacts will use placeholder if generation is needed.')

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 9: Run Case Study Pipeline (Orchestrator Mode)

The runner will:
1. Select best samples based on prediction + artifact availability
2. Generate any missing Grad-CAM, Attention, SHAP, LIME artifacts
3. Create combined multi-method figures
4. Generate metadata + analysis text

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 9 — Run Pipeline (Orchestrator Mode)')
print('='*60)

runner = CaseStudyRunner(
    exp_dir=EXP_DIR,
    dataset_csv=DATASET_CSV,
    image_dir=IMAGE_DIR,
    split=SPLIT,
    model=model,
    tokenizer=tokenizer,
    image_processor=image_processor,
    device=device,
    shap_background=shap_background,
)

results = runner.run()

print(f'\nPipeline complete ({time.time()-t0:.1f}s)')

### STEP 10: Selection Summary

In [ ]:
print('='*60)
print('  Phase 6 — Step 10 — Selection Summary')
print('='*60)

selections = results.get('selections', {})
total = sum(len(v) for v in selections.values())
print(f'\n{"Case Type":<18s} {"Count":>5s}')
print('-'*25)
for ct in ['conflict', 'high_error', 'text_dominant', 'image_dominant', 'difficult', 'agreement', 'correct']:
    print(f'{ct:<18s} {len(selections.get(ct, [])):5d}')
print(f'{"TOTAL":<18s} {total:5d}')

### STEP 11: Display Example Combined Figure

In [ ]:
print('='*60)
print('  Phase 6 — Step 11 — Example Figure')
print('='*60)

example_fig = None
for root, dirs, files in os.walk(XAI_OUT_DIR):
    for f in files:
        if f.startswith('combined_figure_') and f.endswith('.png'):
            example_fig = os.path.join(root, f)
            break
    if example_fig: break

if example_fig:
    img = PILImage.open(example_fig)
    fig, ax = plt.subplots(1, 1, figsize=(12, 14))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Example Combined Figure', fontsize=14)
    plt.tight_layout(); plt.show()
    print(f'Displayed: {example_fig}')
else:
    print('No combined figures generated.')

### STEP 12: Validation + Final Summary

In [ ]:
print('='*60)
print('  PHASE 6 CASE STUDY — FINAL SUMMARY')
print('='*60)

artifact_count = sum(len(f) for _, _, f in os.walk(XAI_OUT_DIR))
case_dirs = [d for d in os.listdir(XAI_OUT_DIR)
             if d.startswith('case_') and os.path.isdir(os.path.join(XAI_OUT_DIR, d))]

print(f'  Experiment      : {EXP_ID}')
print(f'  Mode            : Orchestrator (auto-generates missing artifacts)')
print(f'  Total cases     : {total}')
print(f'  Case directories: {len(case_dirs)}')
print(f'  Total artifacts : {artifact_count}')
print(f'  Output dir      : {XAI_OUT_DIR}')
print()

checks = [
    ('Selection completed',  total > 0),
    ('Manifest CSV',        os.path.isfile(os.path.join(XAI_OUT_DIR, 'sample_manifest.csv'))),
    ('Index CSV',           os.path.isfile(os.path.join(XAI_OUT_DIR, 'case_study_index.csv'))),
    ('Summary MD',          os.path.isfile(os.path.join(XAI_OUT_DIR, 'case_study_summary.md'))),
    ('Selection log',       os.path.isfile(os.path.join(XAI_OUT_DIR, 'selection_log.json'))),
    ('Case dirs exist',     len(case_dirs) > 0),
]

# Check generation log
gen_log = os.path.join(XAI_OUT_DIR, 'generation_log.json')
if os.path.isfile(gen_log):
    with open(gen_log, 'r') as f:
        gl = json.load(f)
    summary = gl.get('summary', {})
    print(f'  Artifact Generation:')
    print(f'    Existing : {summary.get("existing", 0)}')
    print(f'    Generated: {summary.get("generated", 0)}')
    print(f'    Failed   : {summary.get("failed", 0)}')
    checks.append(('Generation log', True))
else:
    checks.append(('Generation log', False))

print()
all_ok = True
for desc, passed in checks:
    s = 'PASSED' if passed else 'FAILED'
    if not passed: all_ok = False
    print(f'  [{s:6s}] {desc}')

print('='*60)
if all_ok:
    print('  All checks PASSED. Phase 6 complete.')
else:
    print('  Some checks FAILED.')
print('='*60)